In [ ]:
from frumkin.solve.bvpsweep import sweep_solve_bvp
from frumkin.tools.mesh import create_semi_infinite_mesh
import numpy as np
from scipy import constants as C 

EPSOPT = 2
KAPPA = C.elementary_charge ** 2 / (C.Boltzmann * 298 * C.angstrom * C.epsilon_0)
CONCENTRATION = 10e-3
NBULK = CONCENTRATION * C.Avogadro * 1000 * (C.angstrom) ** 3  # dimensionless, particles/A^3 but A=1
BULK_FRAC = 2 * NBULK * 8
K1 = 0
K2 = -0.15
K3 = 0.2
P0 = 0 * 1e-2 * KAPPA * C.angstrom ** 2 / C.elementary_charge

def densities(y):
    cation_boltzmann_fac = np.exp(-y[0, :])
    anion_boltzmann_fac = np.exp(y[0, :])
    denom = (1 - BULK_FRAC) + BULK_FRAC * np.cosh(y[0, :])
    cation_density = NBULK * cation_boltzmann_fac / denom
    anion_density = NBULK * anion_boltzmann_fac / denom
    return cation_density, anion_density

def rhs(x, y):
    dy0 = y[1, :]
    npos, nneg = densities(y)
    dy1 = 1/EPSOPT * (y[3, :] - KAPPA * (npos - nneg))
    dy2 = y[3, :]
    dy3 = y[4, :]
    dy4 = y[5, :]
    dy5 = 1/K3 * (-y[1, :] - K1 * y[2, :] + K2 * y[4, :])
    return np.vstack([dy0, dy1, dy2, dy3, dy4, dy5]) #, dy2, dy3, dy4, dy5])

def bc(ya, yb, p):
    return np.array(
        [
            ya[1] - (P0 - p),
            yb[0],
            ya[2] - P0,
            yb[2],
            yb[3],
            yb[4],
        ]
    )

x0 = create_semi_infinite_mesh(xmax=1000, n_points=1000)
y0 = np.zeros((6, len(x0)))
surface_charge = np.concatenate([np.linspace(-10, 0, 50), np.linspace(0, 10, 50)[1:]])

sol = sweep_solve_bvp(
    rhs,
    bc,
    x0,
    y0,
    sweep_par=surface_charge,
    sweep_par_start=0,
)

sol.shape

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 
fig = plt.figure()
ax = fig.add_subplot()

for i in range(len(surface_charge)):
    ax.plot(x0, sol[0, i, :])

ax.set_xlim([0, 10])
ax.set_xlabel(r"z / $\mathrm{\AA}$")
ax.set_ylabel(r"$e_0\phi/k_\mathrm{B}T$")

In [ ]:
potential = sol[0, :, 0]
capacitance = np.gradient(surface_charge, edge_order=2) / np.gradient(potential)

fig = plt.figure()
ax = fig.add_subplot()
ax.plot(potential, capacitance, '.')

In [ ]:
import numpy as np
from scipy.optimize import root_scalar

def langevin(x):
    """Langevin function L(x) = coth(x) - 1/x, with stable handling near x=0."""
    x = np.asarray(x)
    L = np.zeros_like(x)
    small = np.abs(x) < 1e-6
    L[small] = x[small] / 3  # series expansion
    L[~small] = 1/np.tanh(x[~small]) - 1/x[~small]
    return L

def invert_field(P, kappa, p, n_s):
    """
    Given polarization vector P_vec, compute (E - grad phi).
    Parameters:
        P_vec : ndarray, shape (3,)
        kappa, p, n_s : scalars
    Returns:
        X_vec = E - grad(phi)
    """
    P_mag = np.linalg.norm(P)
    if P_mag == 0:
        return np.zeros_like(P)
    
    C = kappa * p * n_s

    # Solve P = C * L(p*X) for X
    def f(X):
        return C * langevin(p * X) - P_mag

    # Bracket root in a reasonable interval (try up to large field)
    sol = root_scalar(f, bracket=[1e-12, 1e3 / p], method='bisect')
    if not sol.converged:
        raise RuntimeError("Inversion did not converge")

    X_mag = sol.root

    # Direction is opposite to P
    X_vec = - (X_mag / P_mag) * P
    return X_vec

# Parameters
kappa = 1.0
p = 0.5
n_s = 1.0
P = np.array([0.2, 0.0, 0.0])

X = invert_field(P, kappa, p, n_s)
print("E - grad(phi) =", X)
